# 04 — Held-out prediction reports
Generate ROC panels, exploratory threshold curves, element-count summaries, per-element AUC and periodic-block distributions from notebook 03's final holdouts. **Inputs:** final metadata, holdout CSVs and matching split tables for both projects. **Outputs:** figures and underlying tables in each project's `results/` directory.

Run cells in order. Metadata selection must identify one final run per model/domain; use explicit `RUN_OVERRIDES` when several candidates exist. Verified compatibility aliases are created for the reporting API. Existing differing aliases raise an error.

Main ROC and element-count MP panels display MP-I; per-element and periodic-block analyses retain all trained domains. Per-element subsets overlap. Thresholds selected with test labels are descriptive operating points, not independently validated choices.


In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from project_config import make_cfg
from private_paths import COD_DIR, MP_DIR
cfg_cod = make_cfg(project_name="COD_CANON_V2", root_cif=COD_DIR)
cfg_mp = make_cfg(project_name="MP_ALL", root_cif=MP_DIR)
PAPER_RUN = "PAPER_V2"
RUN_OVERRIDES = {}  # (project_name, model, domain): exact run_id if selection is ambiguous.
runs = {model: f"{PAPER_RUN}__final__{model}__magpie" for model in ("rf", "lgbm")}

import pandas as pd
from IPython.display import display
from reports.paper_inputs import prepare_report_inputs
from reports.report_data import prepare_reports_data
from reports.holdout_threshold import plot_threshold_metrics
from reports.stats_by_n_elements import plot_stats_by_n_elements, plot_stats_by_n_elements_legend
from reports.metric_vs_Z import plot_metric_vs_Z_by_run_and_domain, plot_metric_vs_Z_legend
from reports.roc_plots import plot_roc_panels_by_dataset
from reports.plot_periodic_block import plot_auc_block_for_runs
from reports.holdout_threshold import save_threshold_legend
models = ["rf", "lgbm"]
domains = ["I", "O", "M"]
selection = prepare_report_inputs([cfg_cod, cfg_mp], runs, overrides=RUN_OVERRIDES,
                                  paper_run=PAPER_RUN, kind="holdout")
display(pd.DataFrame(selection))


## Assemble reporting data
Load holdouts and attach composition metadata from the corresponding project.


In [ ]:

# Prepare base dataframes (COD, MP, stacked, cleaned)
data = prepare_reports_data(cfg_cod, cfg_mp, runs=runs, models=models, domains=domains)

df_cod = data["df_cod"]
df_mp = data["df_mp"]
combined_holdouts = data["combined_holdouts"]
df_clean_all = data["df_clean_all"]

# Display prediction summaries used by subsequent figures.
print("COD holdouts (head):")
display(df_cod.head())
print("MP holdouts (head):")
display(df_mp.head())
print("Combined combined_holdouts by source/domain:")
display(combined_holdouts.groupby(["source", "domain"]).size())
print("df_clean_all by source/domain:")
display(df_clean_all.groupby(["source", "domain"]).size())


## ROC panels — Figure 1
ROC-AUC uses positive-class probabilities; each curve corresponds to one fitted final model.


In [ ]:
plot_roc_panels_by_dataset(
    df_clean_all,
    runs=runs,
    models=[
        "rf",
        "lgbm"
    ],
    domains=[
        "I",
        "M",
        "O"
    ],
    dataset_col="source",
    save=True,
    project_dir=cfg_mp.PROJECT_DIR,
    verbose=True,
    title = "",
)


## Per-element AUC — Figure 4 and Figures S1–S2
Evaluate records containing each element, requiring at least 30 records and both target classes. The historical reporting element map ends at uranium (Z=92); later elements are not represented. The shared legend is saved separately. Each curve describes one final model, without aggregation across independently fitted runs.


In [ ]:
# Plot per-element metrics without element-symbol annotations.
plot_metric_vs_Z_by_run_and_domain(df_cod, runs=runs, models=models, domains=domains, metric="auc",
                                   project_dir=cfg_cod.PROJECT_DIR, save=True, title="", annotate=False, show_legend=False)


In [ ]:
# Plot per-element metrics without element-symbol annotations.
plot_metric_vs_Z_by_run_and_domain(df_mp, runs=runs, models=models, domains=domains, metric="auc",
                                   project_dir=cfg_mp.PROJECT_DIR, save=True, title="", annotate=False, show_legend=False)


In [ ]:
plot_metric_vs_Z_legend(project_dir=cfg_cod.PROJECT_DIR, run_id="paper",
                        name="metric_vs_Z_legend", save=True)


In [ ]:
# Persist the numerical per-element inputs required by periodic-block reports.
from reports.per_element import per_element_metric_table_Z_by_run_and_domain
for cfg, frame in ((cfg_cod, df_cod), (cfg_mp, df_mp)):
    per_element_metric_table_Z_by_run_and_domain(frame, runs=runs, models=models,
        domains=domains, metric="auc", min_n=30, project_dir=cfg.PROJECT_DIR, save=True)


## Number of distinct elements — Figure 3
Bins are 1–2, 3, 4 and 5+ elements. Histogram counts are shown once per dataset/domain and are not summed across models.


In [ ]:
datasets=["MP", "COD"]

plot_stats_by_n_elements(
    df_clean_all,
    runs=runs,
    domains=[
        "I",
        "O",
        "M"
    ],
    datasets=datasets,
    metrics=["auc"],
    overlay=True,
    save=True,
    project_dir=cfg_mp.PROJECT_DIR,
    show_hist=True,
    hist_palette="green",
    show_legend=True,
    title = "",
)


In [ ]:
fig_legend = plot_stats_by_n_elements_legend(project_name="MP_ALL",
    project_dir=cfg_mp.PROJECT_DIR, save=True, run_id="paper",
    name="stats_by_n_elements_legend", hist_palette="green")
display(fig_legend)


## Periodic-block distributions — Figure 5
Group element-specific RF AUC values by the element's s/p/d/f block. Boxes summarize variation across these AUC values, not distinct sets of crystals or bootstrap uncertainty. All three domains are retained for both projects. The input per-element tables already impose the 30-record threshold.


In [ ]:
summaries = plot_auc_block_for_runs(
    cfg=cfg_mp,
    runs={"rf": runs["rf"]},
    domains=domains,
    min_n=20,
    overlay=True,
    save=True,
    title="",
    verbose=True,
)


In [ ]:
summaries = plot_auc_block_for_runs(
    cfg=cfg_cod,
    runs={"rf": runs["rf"]},
    domains=domains,
    min_n=20,
    overlay=True,
    save=True,

    title="",
    verbose=True,
)


## Exploratory threshold curves — Figure 2
Generate threshold curves for all model/domain combinations, marking a target recall of 0.8. The manuscript uses RF inorganic panels. Save the numerical operating-point summary separately from the fixed-threshold test metrics.


In [ ]:
# Run threshold metrics plots for all (project, model, domain) combinations used in this notebook

experiments = [
    ("COD_CANON_V2", cfg_cod, df_cod),
    ("MP_ALL", cfg_mp, df_mp),
]
models_to_eval = ["rf", "lgbm"]
# Domains as encoded in dataframes (I/O/M mapping to N/O/M in some contexts is handled upstream)
domains_to_eval = ["I", "O", "M"]

rows_thr = []
shared_handles = None
shared_labels = None

for proj_name, cfg, df_src in experiments:
    for model in models_to_eval:
        if model not in runs:
            continue
        run_id = runs[model]
        for dom in domains_to_eval:
            sub = df_src[(df_src["model"] == model) & (df_src["domain"] == dom)]
            if sub.empty:
                continue
            fig, ax, stats = plot_threshold_metrics(
                y_true=sub["y"].to_numpy(),
                y_prob=sub["y_prob"].to_numpy(),
                target_recall=0.8,
                title="",
                show_legend=False,
            )
            # Capture legend handles/labels once for a shared legend figure
            if shared_handles is None:
                shared_handles, shared_labels = ax.get_legend_handles_labels()

            # Save figure under <PROJECT_DIR>/results/paper/figures
            proj_dir = Path(cfg.PROJECT_DIR)
            out_dir = proj_dir / "results" / "paper" / "figures"
            out_dir.mkdir(parents=True, exist_ok=True)
            fname = f"threshold_metrics__proj_{proj_name}__model_{model}__domain_{dom}.png"
            fig.savefig(out_dir / fname, dpi=200, bbox_inches="tight")

            display(fig)
            rows_thr.append(
                {
                    "project": proj_name,
                    "model": model,
                    "run_id": run_id,
                    "domain": dom,
                    **stats,
                }
            )

threshold_summary = pd.DataFrame(rows_thr)
print("\nThreshold metrics summary:")
display(threshold_summary)

for cfg in (cfg_cod, cfg_mp):
    destination = Path(cfg.RESULTS_DIR) / "paper" / "tables"
    destination.mkdir(parents=True, exist_ok=True)
    threshold_summary[threshold_summary["project"] == cfg.PROJECT_NAME].to_csv(destination / "threshold_summary.csv", index=False)


In [ ]:
# Shared legend for threshold metrics (one figure, reusable across plots)

save_threshold_legend(cfg_cod.PROJECT_DIR)
save_threshold_legend(cfg_mp.PROJECT_DIR)
